# Beat the Streak — First ML Model

**Goal:** Predict `got_hit` (binary) from features stored in the `Prediction` table.

Steps:
1. Load data from Django DB
2. Parse `score_breakdown` for raw features
3. Build feature matrix
4. Time-based train/test split
5. Logistic regression baseline
6. Random forest
7. Compare vs. existing hand-crafted `score`

In [ ]:
import sys
import os

# Point to project root so Django can find settings
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')

import django
django.setup()

print('Django ready')

In [ ]:
import re
import numpy as np
import pandas as pd
from predictor.models import Prediction

# Load all rows that have a resolved outcome
qs = Prediction.objects.exclude(got_hit=None).values(
    'date', 'name', 'bats', 'list_type',
    'score',
    'pitcher_hand', 'pitcher_era', 'pitcher_whip',
    'is_home', 'avg_batting_order', 'park_factor',
    'team_rank', 'hit_streak',
    'score_breakdown',
    'got_hit',
)

df = pd.DataFrame.from_records(qs)
print(f'Loaded {len(df)} rows across {df["date"].nunique()} dates')
print(f'Hit rate: {df["got_hit"].mean():.1%}')
df.head(3)

## Parse `score_breakdown` for raw features

The breakdown string looks like:
```
P(Hit): 74.2% | BA: 0.285 | xBA: 0.310 | HH%: 42% | K%: 15.2% (+0.5σ) | BB%: 8.1% | Speed: 28.3 ft/s | ...
```

In [ ]:
def parse_breakdown(s):
    """Extract numeric features from score_breakdown string."""
    result = {}
    if not s:
        return result

    patterns = {
        'ba':       r'BA:\s*([0-9.]+)',
        'xba':      r'xBA:\s*([0-9.]+)',
        'hh_pct':   r'HH%:\s*([0-9.]+)',
        'k_pct':    r'K%:\s*([0-9.]+)',
        'bb_pct':   r'BB%:\s*([0-9.]+)',
        'speed':    r'Speed:\s*([0-9.]+)',
        'p_era':    r'P\.ERA:\s*([0-9.]+)',
        'p_whip':   r'P\.WHIP:\s*([0-9.]+)',
        'p_k9':     r'P\.K/9:\s*([0-9.]+)',
    }

    for key, pat in patterns.items():
        m = re.search(pat, s)
        result[key] = float(m.group(1)) if m else np.nan

    return result

parsed = df['score_breakdown'].apply(parse_breakdown)
parsed_df = pd.DataFrame(parsed.tolist(), index=df.index)

print('Parsed features coverage (non-null %)')
print((parsed_df.notna().mean() * 100).round(1).to_string())

## Build feature matrix

In [ ]:
df = pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

# Encode categoricals
df['bats_L']    = (df['bats'] == 'L').astype(int)
df['bats_S']    = (df['bats'] == 'S').astype(int)   # switch
df['pitch_R']   = (df['pitcher_hand'] == 'R').astype(int)
df['pitch_L']   = (df['pitcher_hand'] == 'L').astype(int)
df['platoon']   = ((df['bats'] == 'L') & (df['pitcher_hand'] == 'R') |
                   (df['bats'] == 'R') & (df['pitcher_hand'] == 'L')).astype(int)
df['is_a_list'] = (df['list_type'] == 'A').astype(int)

FEATURE_COLS = [
    # Hand-crafted model output (should be useful but the model should learn beyond it)
    'score',
    # Raw batting stats
    'ba', 'xba', 'hh_pct', 'k_pct', 'bb_pct', 'speed',
    # Game context
    'is_home', 'park_factor', 'avg_batting_order', 'hit_streak',
    # Pitcher features
    'pitcher_era', 'pitcher_whip', 'p_era', 'p_whip', 'p_k9',
    # Team context
    'team_rank',
    # Encoded categoricals
    'bats_L', 'bats_S', 'pitch_R', 'pitch_L', 'platoon', 'is_a_list',
]

X = df[FEATURE_COLS].copy()
y = df['got_hit'].astype(int)

print(f'Feature matrix: {X.shape}')
print(f'Class balance: {y.mean():.1%} hits')
X.describe().round(3)

## Time-based train/test split

**Critical:** Never use random split on time-series data. Train on earlier dates, test on later dates.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df_sorted = df.sort_values('date').reset_index(drop=True)
X = df_sorted[FEATURE_COLS].copy()
y = df_sorted['got_hit'].astype(int)

# 80/20 time split
split_idx = int(len(df_sorted) * 0.8)
split_date = df_sorted.iloc[split_idx]['date']

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {len(X_train)} rows up to {df_sorted.iloc[split_idx-1]["date"].date()}')
print(f'Test:  {len(X_test)} rows from {split_date.date()}')
print(f'Train hit rate: {y_train.mean():.1%}  |  Test hit rate: {y_test.mean():.1%}')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Preprocessing: impute NaNs with median, then scale
preprocessor = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

## Baseline: Hand-crafted score as sole predictor

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score
import matplotlib.pyplot as plt

# Treat existing score (0-100) as a probability proxy
score_proba = X_test[['score']].values / 100.0
baseline_auc = roc_auc_score(y_test, score_proba)
print(f'Baseline (hand-crafted score) AUC: {baseline_auc:.4f}')

## Model 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

lr_pipeline.fit(X_train, y_train)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_proba)

print(f'Logistic Regression AUC: {lr_auc:.4f}')
print()
print(classification_report(y_test, lr_pipeline.predict(X_test), target_names=['No hit', 'Hit']))

## Model 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('pre', Pipeline([('impute', SimpleImputer(strategy='median'))])),  # RF doesn't need scaling
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)),
])

rf_pipeline.fit(X_train, y_train)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)

print(f'Random Forest AUC: {rf_auc:.4f}')
print()
print(classification_report(y_test, rf_pipeline.predict(X_test), target_names=['No hit', 'Hit']))

## Results comparison

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(y_test, score_proba.ravel(), name=f'Hand-crafted score (AUC={baseline_auc:.3f})', ax=ax)
RocCurveDisplay.from_predictions(y_test, lr_proba, name=f'Logistic Regression (AUC={lr_auc:.3f})', ax=ax)
RocCurveDisplay.from_predictions(y_test, rf_proba, name=f'Random Forest (AUC={rf_auc:.3f})', ax=ax)

ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_title('ROC Curves — Beat the Streak Models')
ax.legend()
plt.tight_layout()
plt.show()

print('\nSummary')
print('-' * 40)
print(f'Hand-crafted score AUC : {baseline_auc:.4f}')
print(f'Logistic Regression AUC: {lr_auc:.4f}  (delta: {lr_auc - baseline_auc:+.4f})')
print(f'Random Forest AUC      : {rf_auc:.4f}  (delta: {rf_auc - baseline_auc:+.4f})')

## Feature importance (Random Forest)

In [ ]:
importances = rf_pipeline.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 8))
feat_imp.plot(kind='barh', ax=ax)
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(feat_imp.tail(10).sort_values(ascending=False).round(4).to_string())

## Logistic Regression coefficients (which direction do features push the model?)

In [ ]:
coef = pd.Series(
    lr_pipeline.named_steps['clf'].coef_[0],
    index=FEATURE_COLS
).sort_values()

fig, ax = plt.subplots(figsize=(7, 8))
coef.plot(kind='barh', ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients (scaled)')
plt.tight_layout()
plt.show()

## Save the best model

Once you have enough data and are happy with a model, save it here.

In [ ]:
import joblib
import os

# Pick the better model
best_model = rf_pipeline if rf_auc >= lr_auc else lr_pipeline
best_name = 'random_forest' if rf_auc >= lr_auc else 'logistic_regression'

models_dir = os.path.join(PROJECT_ROOT, 'data', 'models')
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, f'hit_predictor_{best_name}.joblib')
joblib.dump({
    'model': best_model,
    'feature_cols': FEATURE_COLS,
    'auc': max(rf_auc, lr_auc),
    'baseline_auc': baseline_auc,
}, model_path)

print(f'Model saved to {model_path}')
print(f'AUC: {max(rf_auc, lr_auc):.4f} vs baseline {baseline_auc:.4f}')